# Voice-Controlled Graph Assistant

Ask natural language questions about the Goodreads book graph using your voice.

## How it works

```
voice-assistant.html (browser)
    │  WebRTC audio
    ▼
LiveKit Cloud
    │
    ▼
LiveKit worker (this notebook)
    ├── VAD  : Silero
    ├── STT  : OpenAI Whisper
    ├── LLM  : GPT-4o-mini -> query_graph tool -> Text2Cypher -> Neo4j Aura
    └── TTS  : OpenAI TTS
```

## Before running the notebook

1. `export` your Neo4j Aura connection URI, username and password.
2. `export` your OpenAI API key.
3. `export` your LiveKit Cloud URL, API key and API secret.
4. In LiveKit Cloud dashboard -> Settings -> enable **Token server**.
   Copy the sandbox ID (`token-server-xxxxxx`) and paste it into `voice-assistant.html`.

## 1. Install dependencies

In [1]:
!pip cache purge --quiet

In [2]:
!pip install neo4j==6.2.0 \
             neo4j_graphrag==1.18.0 \
             pandas==3.0.3 \
             openai==2.2.0 \
             livekit==1.0.16 \
             livekit-agents==1.2.14 \
             livekit-api==1.0.6 \
             livekit-plugins-openai==1.2.14 \
             livekit-plugins-silero==1.2.14 \
             livekit-protocol==1.0.8 \
             livekit-blingfire==1.0.0 \
             av==15.1.0 \
             onnxruntime==1.23.1 \
             --no-deps --quiet

print("Install complete.")

Install complete.


In [3]:
import asyncio
import logging
import os
import sys

from livekit.agents import Worker, WorkerOptions
from neo4j import GraphDatabase
from neo4j_graphrag.llm import OpenAILLM
from neo4j_graphrag.retrievers import Text2CypherRetriever

In [4]:
logging.basicConfig(level = logging.ERROR, stream = sys.stdout, force = True)
logging.getLogger("graph-agent").setLevel(logging.INFO)
logging.getLogger("graph-agent-core").setLevel(logging.INFO)

logger = logging.getLogger("graph-agent")

## 2. Credentials

In [5]:
# Neo4j Aura
NEO4J_URI      = os.environ["NEO4J_URI"]
NEO4J_USER     = os.environ["NEO4J_USER"]
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]

# OpenAI
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

# LiveKit Cloud
LIVEKIT_URL        = os.environ["LIVEKIT_URL"]
LIVEKIT_API_KEY    = os.environ["LIVEKIT_API_KEY"]
LIVEKIT_API_SECRET = os.environ["LIVEKIT_API_SECRET"]

print("Credentials set.")

Credentials set.


## 3. Neo4j connection

In [6]:
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth = (NEO4J_USER, NEO4J_PASSWORD)
)

print(driver.verify_connectivity())
print("Connection created.")

None
Connection created.


## 4. LLM and retriever

In [7]:
llm = OpenAILLM(
    model_name = "gpt-4o-mini",
    model_params = {"temperature": 0}
)

schema = """
You are querying a Goodreads book graph.

Nodes:

Author:
- name
- average_rating

Book:
- title
- publication_year
- average_rating
- ratings_count

Relationships:

(:Author)-[:AUTHORED]->(:Book)
(:Book)-[:SIMILAR_TO]->(:Book)

Important rules:

1. For highest rated books, query Book.average_rating directly.
   Do not use Author.average_rating.

2. For book similarity, use SIMILAR_TO relationships.

3. Do not create relationships that are not listed.

4. Always return properties, not entire nodes.

5. Always alias every returned property using AS.
   Example: RETURN b.title AS title, b.average_rating AS average_rating

6. Always filter out null values before sorting.
   Example: WHERE b.average_rating IS NOT NULL

Cypher syntax reminders:
- To find the top N results by a property, use ORDER BY and LIMIT, not subqueries.
- Never use SQL syntax such as SELECT, MAX(), FROM or WHERE with subqueries.
- Example for books by an author: MATCH (a:Author {name: "Name"})-[:AUTHORED]->(b:Book) RETURN b.title AS title, b.publication_year AS publication_year
- Example for highest rated: MATCH (b:Book) WHERE b.average_rating IS NOT NULL RETURN b.title AS title, b.average_rating AS average_rating ORDER BY b.average_rating DESC LIMIT 10
- Example for similar books: MATCH (b:Book {title: "Title"})-[:SIMILAR_TO]->(s:Book) RETURN s.title AS title, s.average_rating AS average_rating
- publication_year is stored as a string. Use toInteger() when sorting or comparing by year. Example: MATCH (b:Book) RETURN MAX(toInteger(b.publication_year)) AS most_recent_year
- average_rating is stored as a string. Use toFloat() when comparing numerically. Example: MATCH (b:Book) WHERE toFloat(b.average_rating) >= 3.5 RETURN b.title AS title, b.average_rating AS average_rating
"""

retriever = Text2CypherRetriever(
    driver = driver,
    llm = llm,
    neo4j_schema = schema
)

print("LLM and retriever ready.")

LLM and retriever ready.


## 5. Exploration

In [8]:
with driver.session() as session:
    result = session.run("""
        MATCH (a:Author)
        RETURN a.name AS author
        LIMIT 10
    """)
    for record in result:
        print(record["author"])

Ronald J. Fields
Anita Diamant
Barbara Hambly
Jennifer Weiner
Nigel Pennick
Alfred J. Church
Michael Halberstam
Rachel Roberts
V.L. Locey
Anton Szandor LaVey


## 6. `ask_graph` (synchronous)

Used for interactive testing in the notebook.

In [9]:
def ask_graph(question):
    print("=" * 80)
    print("QUESTION:", question)
    print()

    try:
        result = retriever.search(query_text=question)
    except Exception as e:
        print(f"Cypher generation failed: {e}")
        return None

    cypher = result.metadata["cypher"]
    print("CYPHER:", cypher)

    data = []
    with driver.session() as session:
        for record in session.run(cypher):
            data.append(record.data())

    if len(data) > 5:
        data = data[:5]

    print("DATA:", data)
    print()

    if not data:
        answer = "I found no matching data in the graph for that question."
    else:
        prompt = f"""
Answer the question using only the structured data provided.

Question:
{question}

Data:
{data}

Rules:
- Do not invent information.
- Do not use information from fields that are not explicitly returned.
- If the data contains only a message field, return that message verbatim.
- Keep the answer concise.
- Write in plain spoken English suitable for text-to-speech.
- Do not use bullet points, markdown or special characters.
- Do not infer relationships or facts that are not present in the data.
"""
        response = llm.invoke(prompt)
        answer = response.content

    print(answer)

## 7. Verify `ask_graph` works

In [10]:
# ask_graph("What books were written by Anita Diamant?")

In [11]:
# ask_graph("What are the highest rated books?")

In [12]:
# ask_graph("Who was the first man on the moon?")

## 8. Agent

Written to `agent_core.py` via `%%writefile` to avoid Jupyter pickling issues.
This cell must be re-run every time the kernel restarts.

In [13]:
%%writefile agent_core.py

import asyncio
import logging
import os

from neo4j import GraphDatabase
from neo4j_graphrag.llm import OpenAILLM
from neo4j_graphrag.retrievers import Text2CypherRetriever

from livekit.agents import Agent, AgentSession, JobContext, RoomInputOptions, function_tool
from livekit.plugins import openai, silero

logger = logging.getLogger("graph-agent-core")
logger.setLevel(logging.INFO)

# Graph schema

SCHEMA = """
You are querying a Goodreads book graph.

Nodes:

Author:
- name
- average_rating

Book:
- title
- publication_year
- average_rating
- ratings_count

Relationships:

(:Author)-[:AUTHORED]->(:Book)
(:Book)-[:SIMILAR_TO]->(:Book)

Important rules:

1. For highest rated books, query Book.average_rating directly.
   Do not use Author.average_rating.

2. For book similarity, use SIMILAR_TO relationships.

3. Do not create relationships that are not listed.

4. Always return properties, not entire nodes.

5. Always alias every returned property using AS.
   Example: RETURN b.title AS title, b.average_rating AS average_rating

6. Always filter out null values before sorting.
   Example: WHERE b.average_rating IS NOT NULL

Cypher syntax reminders:
- To find the top N results by a property, use ORDER BY and LIMIT, not subqueries.
- Never use SQL syntax such as SELECT, MAX(), FROM or WHERE with subqueries.
- Example for books by an author: MATCH (a:Author {name: "Name"})-[:AUTHORED]->(b:Book) RETURN b.title AS title, b.publication_year AS publication_year
- Example for highest rated: MATCH (b:Book) WHERE b.average_rating IS NOT NULL RETURN b.title AS title, b.average_rating AS average_rating ORDER BY b.average_rating DESC LIMIT 10
- Example for similar books: MATCH (b:Book {title: "Title"})-[:SIMILAR_TO]->(s:Book) RETURN s.title AS title, s.average_rating AS average_rating
- publication_year is stored as a string. Use toInteger() when sorting or comparing by year. Example: MATCH (b:Book) RETURN MAX(toInteger(b.publication_year)) AS most_recent_year
- average_rating is stored as a string. Use toFloat() when comparing numerically. Example: MATCH (b:Book) WHERE toFloat(b.average_rating) >= 3.5 RETURN b.title AS title, b.average_rating AS average_rating
"""

# Helpers

def clean_for_voice(text):
    """Strip markdown and limit length so TTS sounds natural."""
    cleaned = (
        text
        .replace("**", "").replace("*", "")
        .replace("```", "").replace("`", "")
        .replace("|", " ").replace("---", "")
    )
    lines = [l.strip() for l in cleaned.splitlines() if l.strip()]
    result = " ".join(lines)
    if len(result) > 800:
        result = result[:800] + "... and more."
    return result

# Graph initialization

def initialize_graph():
    driver = GraphDatabase.driver(
        os.environ["NEO4J_URI"],
        auth = (os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"])
    )
    llm = OpenAILLM(
        model_name = "gpt-4o-mini",
        model_params = {"temperature": 0}
    )
    retriever = Text2CypherRetriever(
        driver = driver,
        llm = llm,
        neo4j_schema = SCHEMA
    )
    logger.info("Graph retriever initialized.")
    return driver, retriever

# Agent

class Assistant(Agent):

    def __init__(self, driver, retriever):
        self._driver = driver
        self._retriever = retriever

        super().__init__(
            instructions = (
                "You are a voice assistant for a Goodreads book graph database. "
                "You have no knowledge of books yourself. "
                "You MUST call the query_graph tool for EVERY question without exception. "
                "Never answer from memory. Never skip the tool call. "
                "Keep answers short and conversational. "
                "Never use markdown - plain spoken sentences only."
            ),
        )

    @function_tool()
    async def query_graph(self, question: str) -> str:
        """Answer a question about books by querying the Goodreads graph.

        Args:
            question: The user's natural language question about books.
        """
        logger.info(f"Query: {question}")

        try:
            # Generate Cypher
            result = await asyncio.to_thread(
                self._retriever.search, question
            )
            cypher = result.metadata["cypher"]
            logger.info(f"Cypher: {cypher}")

            # Execute Cypher
            data = []
            with self._driver.session() as session:
                for record in session.run(cypher):
                    data.append(record.data())

            # Limit results before returning to the LLM
            if len(data) > 5:
                data = data[:5]

            if not data:
                return "I found no matching data in the graph for that question."

            logger.info(f"Data: {data}")
            return clean_for_voice(str(data))

        except Exception as e:
            logger.error(f"Query error: {type(e).__name__}: {e}")
            return "I encountered an error querying the graph."

# Entrypoint

async def entrypoint(ctx: JobContext):
    logger.info(f"Job started for room: {ctx.room.name}")

    driver, retriever = initialize_graph()

    await ctx.connect()
    logger.info(f"Connected to room: {ctx.room.name}")

    session = AgentSession(
        stt = openai.STT(model = "whisper-1"),
        llm = openai.LLM(model = "gpt-4o-mini"),
        tts = openai.TTS(model = "gpt-4o-mini-tts"),
        vad = silero.VAD.load(),
    )

    disconnected_future = asyncio.get_event_loop().create_future()

    @ctx.room.on("disconnected")
    def on_disconnected():
        logger.info("Room disconnected")
        if not disconnected_future.done():
            disconnected_future.set_result(True)

    @ctx.room.on("participant_disconnected")
    def on_participant_disconnected(participant):
        if not ctx.room.remote_participants and not disconnected_future.done():
            logger.info("Caller left - ending session")
            disconnected_future.set_result(True)

    await session.start(
        agent = Assistant(driver, retriever),
        room = ctx.room,
        room_input_options = RoomInputOptions(),
    )

    await session.say(
        "Hello! Ask me anything about the book graph, such as which books "
        "an author wrote, or the highest rated books.",
        allow_interruptions = True,
    )

    await disconnected_future

    driver.close()
    logger.info("Session finished.")

Overwriting agent_core.py


## 9. Run the worker

Run this cell, then open `voice-assistant.html` in a browser tab.
Paste in your LiveKit token server sandbox ID and click **Start Call**.

Interrupt the kernel to stop the worker.

In [14]:
from agent_core import entrypoint

async def main_worker():
    logger.info("Starting LiveKit worker...")
    opts = WorkerOptions(
        entrypoint_fnc = entrypoint,
        ws_url = LIVEKIT_URL,
        api_key = LIVEKIT_API_KEY,
        api_secret = LIVEKIT_API_SECRET,
    )
    worker = Worker(opts)
    worker_task = asyncio.create_task(worker.run())
    try:
        await worker_task
    except asyncio.CancelledError:
        pass
    finally:
        logger.info("Worker stopped.")

In [15]:
# Run the worker - interrupt the kernel to stop
await main_worker()

INFO:graph-agent:Starting LiveKit worker...
INFO:graph-agent-core:Job started for room: voice-controlled-wwq03zji
INFO:graph-agent-core:Graph retriever initialized.
INFO:graph-agent-core:Connected to room: voice-controlled-wwq03zji
INFO:graph-agent-core:Query: How many books are in the database?
INFO:graph-agent-core:Cypher: MATCH (b:Book) RETURN COUNT(b) AS book_count
INFO:graph-agent-core:Data: [{'book_count': 10000}]
INFO:graph-agent-core:Query: How many authors are in the database?
INFO:graph-agent-core:Cypher: MATCH (a:Author) RETURN COUNT(a) AS author_count
INFO:graph-agent-core:Data: [{'author_count': 12371}]
INFO:graph-agent-core:Query: Who wrote The Hobbit?
INFO:graph-agent-core:Cypher: MATCH (a:Author)-[:AUTHORED]->(b:Book {title: "The Hobbit"}) RETURN a.name AS author_name
INFO:graph-agent-core:Data: [{'author_name': 'J.R.R. Tolkien'}]
INFO:graph-agent-core:Query: What books are similar to The Hobbit?
INFO:graph-agent-core:Cypher: MATCH (b:Book {title: "The Hobbit"})-[:SIMIL

## 10. Teardown

Run after interrupting the worker.

In [16]:
driver.close()
print("Driver closed.")

Driver closed.
